In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Update with your username and password
username = "aacuser"
password = "password123" 

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# Drop the '_id' column because it causes the datatable to crash
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare's logo
image_filename = 'Grazioso Salvare Logo.png' 
try:
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())
    img_src = 'data:image/png;base64,{}'.format(encoded_image.decode())
except FileNotFoundError:
    img_src = ''
    print(f"Warning: {image_filename} not found. Logo will not display.")


app.layout = html.Div([
    # Logo and Unique Identifier
    html.Center([
        html.Img(src=img_src, style={'width': '250px', 'padding-bottom': '20px'}),
        html.B(html.H1('SNHU CS-340 Dashboard')),
        html.H3('Developed by: Sulav Uprety') # FIX ME: Insert your name here
    ]),
    html.Hr(),
    
    # Interactive filtering options (Radio Buttons)
    html.Div(
        className='row',
        style={'display': 'flex', 'justifyContent': 'center', 'padding': '10px'},
        children=[
            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label': ' Water Rescue ', 'value': 'Water'},
                    {'label': ' Mountain or Wilderness Rescue ', 'value': 'Mountain'},
                    {'label': ' Disaster or Individual Tracking ', 'value': 'Disaster'},
                    {'label': ' Reset ', 'value': 'Reset'}
                ],
                value='Reset',
                labelStyle={'display': 'inline-block', 'margin': '0 15px'}
            )
        ]
    ),
    html.Hr(),
    
    # Interactive Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        
        # Table features for user-friendliness
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'} # Allows scrolling if too many columns
    ),
    html.Br(),
    html.Hr(),
    
    # Side-by-side charts layout
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            style={'flex': '1'}
            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={'flex': '1'}
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Filter the data table based on the radio button selection
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # Determine the query based on the Rescue Type table specifications
    if filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_type == 'Mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_type == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0}
        }
    else: # Reset
        query = {}
        
    # Query the database
    filtered_data = list(db.read(query))
    
    # Rebuild the DataFrame and drop _id
    dff = pd.DataFrame.from_records(filtered_data)
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
        
    # Return dictionary records for the datatable
    return dff.to_dict('records')


# Display the breeds of animal based on quantity represented in the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return []
        
    dff = pd.DataFrame.from_dict(viewData)
    
    # Pie chart showing the distribution of breeds in the current view
    return [
        dcc.Graph(            
            figure = px.pie(dff, names='breed', title='Percentage of Breeds in View')
        )    
    ]
    
# Highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Update the geo-location chart for the selected data entry
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Default to the first row if none is selected
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]
        
    # Ensure the row index is within bounds (in case the table shrinks after filtering)
    if row >= len(dff):
        row = 0
        
    # Austin TX is at [30.75,-97.48]
    # Note: Using location_lat and location_long string matching if your columns are named differently.
    # The template uses iloc[row, 13] and iloc[row, 14], which strictly depends on column order.
    # We will use the template's exact logic here, but double check your data frame column indexes.
    return [
        dl.Map(style={'width': '100%', 'height': '400px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H3("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]

# Run app and display result
app.run_server()

Dash app running on https://africaneptune-lunchisabel-3000.codio.io/proxy/8050/
